## Find the district name based on the latitude and longitude of the earthquake epicenter using the Nominatim API from OpenStreetMap.

## NOTE BELOW ARE ARCHIEVED FILES. DO NOT USE THEM. THEY ARE KEPT FOR REFERENCE PURPOSES ONLY.<br/> AND IF U SEE UNDEFINED VARIABLES, THEY ARE DEFINED IN THE MAIN FILE. DO NOT WORRY ABOUT THEM.

In [ ]:
from enum import Enum
class Features(Enum):
	TIME = 'time'
	LATITUDE = 'latitude'
	LONGITUDE = 'longitude'
	DEPTH = 'depth'
	MAGNITUDE = 'mag'
	MAGNITUDE_TYPE = 'magType'
	NSTATIONS = 'nst'
	GAP = 'gap'
	DMIN = 'dmin'
	RMS = 'rms'
	HORIZONTAL_ERROR = 'horizontalError'
	DEPTH_ERROR = 'depthError'
	MAGNITUDE_ERROR = 'magError'
	MAGNITUDE_NSTATIONS = 'magNst'	

	DISTRICT = 'district'

df_filtered = df_original[[f.value for f in Features if f != Features.DISTRICT]]
df_filtered.head()

df_filtered.to_csv(FILTERED_DATA_PATH, index=False)

In [ ]:

import requests
import time
import pandas as pd

class DistrictFinder:
	URL = "https://nominatim.openstreetmap.org/reverse"
	HEADERS = {
		"User-Agent": "my-nepal-district-app/1.0"
	}
	COUNTRY = "Nepal"
	SLEEP_TIME = 1  # Time to sleep between requests in seconds,to avoid hitting the rate limit of the API
	@staticmethod
	def get_district(lat, lon):
		url = DistrictFinder.URL
		headers = DistrictFinder.HEADERS
		params = {
			"lat": lat,
			"lon": lon,
			"format": "jsonv2",
			"addressdetails": 1,
			"accept-language": "en"
		}

		response = requests.get(
			url,
			params=params,
			headers=headers
		)

		data = response.json()

		address = data.get("address", {})
		country = address.get("country")
		district = address.get("county", "Unknown").replace(" District", "").strip()

		if country != DistrictFinder.COUNTRY:
			return "Unknown"
		else:
			return district
	@staticmethod

	def get_districts(latitudes:pd.Series , longitudes:pd.Series)-> pd.Series:
		districts = []
		for lat, lon in zip(latitudes, longitudes):
			district = DistrictFinder.get_district(lat, lon)
			districts.append(district)
			time.sleep(DistrictFinder.SLEEP_TIME) 
		return  pd.Series(districts, index=latitudes.index)

In [ ]:
import os
import pandas as pd

df_district = df_filtered.copy()

total_records = len(df_district)

print("Fetching districts for the earthquake data...\n"
	  "Grab a cup of coffee, this may take a while depending on the number of records.\n"
	  f"Estimated time: ~1 second per record. For {total_records} records," 
	  f" it will take {total_records // 60} minutes and {total_records % 60} seconds. at best\n"
	  "Most likely will take 2x of the extimated time due to network latency and API response time.\n")

TEMP_CONVERSION_COUNTING_FILE = 'data/temp_conversion_counting.txt'

# Make sure the target column exists, can  inject data into it
if Features.DISTRICT.value not in df_district.columns:
	df_district[Features.DISTRICT.value] = None

current_chunk_index = 0

# Check if we are resuming from a previous run
try:
	with open(TEMP_CONVERSION_COUNTING_FILE, 'r') as f:
		current_chunk_index = int(f.read().strip())
	
	# If resuming, load the existing backup CSV so we don't lose previous chunk data
	if current_chunk_index > 0 and os.path.exists(DISTRICT_DATA_PATH):
		df_district = pd.read_csv(DISTRICT_DATA_PATH)
		print(f"Resuming from saved backup: {DISTRICT_DATA_PATH}")
except FileNotFoundError:
	current_chunk_index = 0



total_records = len(df_district)
chunk_size = 25 
total_chunk_count = total_records // chunk_size + (1 if total_records % chunk_size != 0 else 0)

print(f"Starting from chunk index: {current_chunk_index}")
print(f"Total chunks: {total_chunk_count}")

# Loop through the remaining chunks
for i in range(current_chunk_index, total_chunk_count):
	print(f"Processing chunk {i + 1} of {total_chunk_count}...")
	
	# Calculate start and end indices for the current chunk
	start_idx = i * chunk_size
	end_idx = min((i + 1) * chunk_size, total_records)
	
	# Slice the latitudes and longitudes for this specific chunk
	chunk_lat = df_district[Features.LATITUDE.value].iloc[start_idx:end_idx]
	chunk_lon = df_district[Features.LONGITUDE.value].iloc[start_idx:end_idx]
	
	# Fetch districts
	chunk_districts = DistrictFinder.get_districts(chunk_lat, chunk_lon)
	
	# Assign the results back to the dataframe
	df_district.loc[df_district.index[start_idx:end_idx], Features.DISTRICT.value] = chunk_districts
	
	# Save the updated dataframe as a backup
	df_district.to_csv(DISTRICT_DATA_PATH, index=False)
	
	# Update the counting file to reflect the completed chunk
	with open(TEMP_CONVERSION_COUNTING_FILE, 'w') as f:
		f.write(str(i + 1))

print("All chunks processed successfully!")

# Clean up the temporary counting file since the process is completely finished
if os.path.exists(TEMP_CONVERSION_COUNTING_FILE):
	os.remove(TEMP_CONVERSION_COUNTING_FILE)

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import lambertw

def extract_seismic_features(
    df: pd.DataFrame, 
    felt_mmi: float = 2.5, 
    region: str = "active"
) -> pd.DataFrame:
    """
    Extracts physical felt-vibration features from earthquake catalog data.
    """
    data = df.copy()
    data['time'] = pd.to_datetime(data['time'])
    data = data.sort_values('time').reset_index(drop=True)
    
    # Regional parameters (USGS ShakeMap)
    if region == "active":
        c1, c2, c3, c4 = 2.08, 1.04, 2.09, 0.0019
    else:  # stable
        c1, c2, c3, c4 = 1.85, 1.15, 1.80, 0.0006
        
    M = data['mag'].values
    h = data['depth'].fillna(10.0).values  # Default to 10km depth if missing
    
    # 1. Near-field saturation radius R0(M)
    R0 = 10 ** (-0.281 + 0.251 * M)
    
    # 2. Transformed Lambert W parameters
    b = c3 / c4
    c = (c1 + c2 * M - felt_mmi) / c4
    
    # 3. Vectorized Lambert W evaluation for R_eff
    ln10 = np.log(10)
    z = (ln10 / b) * (10.0 ** (c / b))
    R_eff = (b / ln10) * lambertw(z).real
    
    # 4. Surface felt radius accounting for depth (h) and saturation (R0)
    # R_jb^2 = R_eff^2 - R0^2 - h^2
    rad_sq = R_eff**2 - R0**2 - h**2
    r_felt = np.where(rad_sq > 0, np.sqrt(rad_sq), 0.0)
    
    # 5. Extract core physical features
    data['r_felt_km'] = r_felt
    data['felt_area_sqkm'] = np.pi * (r_felt ** 2)
    
    # 6. Peak intensity directly above hypocenter (R_jb = 0)
    R_eff_epicenter = np.sqrt(R0**2 + h**2)
    data['max_nearfield_mmi'] = c1 + c2 * M - c3 * np.log10(R_eff_epicenter) - c4 * R_eff_epicenter
    
    return data

# Example Usage with catalog data:
# df = pd.read_csv("earthquakes.csv")
# df_featured = extract_seismic_features(df)